# Regression

This notebook handles the reproduction of the regression part of the paper. This includes:
- recreating the R workflow in python as closely as possible
- QR matrix based decomposition
- finding the optimal ntree
- fitting random forest models
- calculcating permutation


In [1]:
import pandas as pd
import numpy as np
from scipy.linalg import qr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load all datasets
raw_allranks = pd.read_excel("..\\data_files\\Raw score data\\allranks_raw.xlsx")
raw_bronze = pd.read_excel("..\\data_files\\Raw score data\\bronze_raw.xlsx")
raw_gold = pd.read_excel("..\\data_files\\Raw score data\\gold_raw.xlsx")
raw_diamond = pd.read_excel("..\\data_files\\Raw score data\\diamond_raw.xlsx")
raw_gc = pd.read_excel("..\\data_files\\Raw score data\\gc_raw.xlsx")

diff_allranks = pd.read_excel("..\\data_files\\Difference score data\\allrank_diff.xlsx")
diff_bronze = pd.read_excel("..\\data_files\\Difference score data\\bronze_diff.xlsx")
diff_gold = pd.read_excel("..\\data_files\\Difference score data\\gold_diff.xlsx")
diff_diamond = pd.read_excel("..\\data_files\\Difference score data\\diamond_diff.xlsx")
diff_gc = pd.read_excel("..\\data_files\\Difference score data\\GC_diff.xlsx")

In [12]:
# copilot was used for the remove_multicollinear_columns_qr function, as I was unsure how to replicate the R logic in Python.
def remove_multicollinear_columns_qr(X, tol=1e-10):
    """
    Remove multicollinear columns using QR decomposition with column pivoting.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix. All columns should be numeric.
    tol : float
        Threshold for detecting linear dependence from the diagonal of R.

    Returns
    -------
    X_reduced : pandas.DataFrame
        DataFrame containing only linearly independent columns.
    kept_columns : list[str]
        Column names kept after QR selection.
    dropped_columns : list[str]
        Column names removed as collinear.
    """
    if not isinstance(X, pd.DataFrame):
        raise TypeError('X must be a pandas DataFrame')

    X_numeric = X.apply(pd.to_numeric, errors='coerce')
    valid = X_numeric.dropna(axis=1, how='any')
    if valid.shape[1] == 0:
        raise ValueError('No fully numeric columns available after coercion')

    matrix = valid.to_numpy(dtype=float)
    _, r, piv = qr(matrix, mode='economic', pivoting=True)
    diag = np.abs(np.diag(r))

    if diag.size == 0:
        kept_idx = np.array([], dtype=int)
    else:
        threshold = tol * diag[0]
        kept_idx = np.where(diag > threshold)[0]

    kept_columns = valid.columns[piv[kept_idx]].tolist()
    dropped_columns = [c for c in valid.columns if c not in kept_columns]

    return valid.loc[:, kept_columns].copy(), kept_columns, dropped_columns


In [14]:
# The authors seems to have tested with a step of 1, however even with a step of 10 the code the optimaization took an hour to run. As such we have opted to make a compromise and use a step of 5.
def find_best_ntree_regression(X, y, max_trees=1000, step=5, random_state=123):
    results = []

    max_features = max(1, X.shape[1] // 3) # This replicates the R default of mtry = p/3 for regression, where p is the number of predictors.

    for n in range(100, max_trees + 1, step):
        rf = RandomForestRegressor(
            n_estimators=n,
            max_features=max_features,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=random_state
        )

        rf.fit(X, y)

        oob_pred = rf.oob_prediction_
        mse = mean_squared_error(y, oob_pred)
        r2 = r2_score(y, oob_pred)

        results.append({
            "ntree": n,
            "oob_mse": mse,
            "oob_r2": r2
        })

    return pd.DataFrame(results).sort_values("oob_mse")

In [ ]:
# This whole section is a loop on the datasets, which applies the functions defined earlier and collects the results in a df

target_col = 'diff.goals.ag'
datasets = [
    ('raw_allranks', raw_allranks),
    ('raw_bronze', raw_bronze),
    ('raw_gold', raw_gold),
    ('raw_diamond', raw_diamond),
    ('raw_gc', raw_gc),
    ('diff_allranks', diff_allranks),
    ('diff_bronze', diff_bronze),
    ('diff_gold', diff_gold),
    ('diff_diamond', diff_diamond),
    ('diff_gc', diff_gc),
]

all_results = {}
best_results_summary = []

for dataset_name, df in datasets:

    candidate_features = df.drop(columns=[target_col])

    X_reduced, kept_columns, dropped_columns = remove_multicollinear_columns_qr(candidate_features, tol=1e-10)

    if dropped_columns:
        print(f"Dropped columns: {dropped_columns}")

    ntree_results = find_best_ntree_regression(X_reduced, df[target_col], max_trees=1000, step=5, random_state=123)

    best_row = ntree_results.iloc[0]
    best_ntree = int(best_row['ntree'])
    best_mse = best_row['oob_mse']
    best_r2 = best_row['oob_r2']

    all_results[dataset_name] = {
        'X_reduced': X_reduced,
        'y': df[target_col],
        'kept_columns': kept_columns,
        'dropped_columns': dropped_columns,
        'ntree_results': ntree_results,
        'best_ntree': best_ntree,
        'best_mse': best_mse,
        'best_r2': best_r2,
    }

    best_results_summary.append({
        'Dataset': dataset_name,
        'Type': dataset_name.split('_', 1)[0],
        'N_Features_Original': candidate_features.shape[1],
        'N_Features_QR': len(kept_columns),
        'N_Dropped': len(dropped_columns),
        'Best_ntree': best_ntree,
        'OOB_MSE': best_mse,
        'OOB_R2': best_r2,
    })

summary_df = pd.DataFrame(best_results_summary)

print("Summary of ntree optimization across all datasets")
print(summary_df.to_string(index=False))

summary_df.to_csv('../outputs/ntree_optimization_summary.csv', index=False)

Summary of ntree optimization across all datasets
      Dataset Type  N_Features_Original  N_Features_QR  N_Dropped  Best_ntree  OOB_MSE   OOB_R2
 raw_allranks  raw                   28             28          0         855 3.611600 0.746904
   raw_bronze  raw                   28             28          0         845 3.908066 0.792571
     raw_gold  raw                   28             28          0         960 3.517468 0.743137
  raw_diamond  raw                   28             28          0         995 3.658571 0.725345
       raw_gc  raw                   27             27          0        1000 4.052708 0.713095
diff_allranks diff                   26             26          0         960 2.295142 0.839126
  diff_bronze diff                   26             26          0        1000 2.991057 0.841057
    diff_gold diff                   26             26          0        1000 2.433283 0.822310
 diff_diamond diff                   26             26          0         995 2.365852

In [ ]:
# Below is an approximation of the R rfPermute's workflow used in the paper.
# Exact reproduction would require matching R randomForest/rfPermute's OOB permutation importance implementation, which scikit-learn implements differently.
# The paper does not explicitly report the number of rfPermute permutations used for regression models, we opted to use 20, the minimum for significant p-values to exist.
n_permutations = 20                  
n_repeats = 2     
n_repeats_NULL = 1

# Since our step for ntree optimazations is too small to capture the exact optimal values reported in the paper, we use the original optimal ntree values from the paper's Figure 6
optimal_ntrees = {
    'raw_allranks': 964,
    'raw_bronze': 856,
    'raw_gold': 386,
    'raw_diamond': 1000,
    'raw_gc': 1000,
    'diff_allranks': 986,
    'diff_bronze': 961,
    'diff_gold': 895,
    'diff_diamond': 989,
    'diff_gc': 991,
}

def fit_dataset_model(dataset_name, random_state=123):
    if dataset_name not in all_results:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    dataset_result = all_results[dataset_name]
    X = dataset_result['X_reduced']
    y = dataset_result['y']
    kept_columns = dataset_result['kept_columns']
    dropped_columns = dataset_result['dropped_columns']
    ntree = optimal_ntrees[dataset_name]

    rf = RandomForestRegressor(
        n_estimators=ntree,
        max_features=max(1, X.shape[1] // 3),
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
    )
    rf.fit(X, y)
    oob_mse = mean_squared_error(y, rf.oob_prediction_)
    oob_r2 = r2_score(y, rf.oob_prediction_)

    return {
        'dataset': dataset_name,
        'ntree': ntree,
        'X': X,
        'y': y,
        'rf': rf,
        'kept_columns': kept_columns,
        'dropped_columns': dropped_columns,
        'model_summary': {
            'Dataset': dataset_name,
            'OOB_MSE': oob_mse,
            'OOB_R2': oob_r2,
        },
    }


The section below is done in separete cells to allow for running one sections at a time

In [17]:
# Fit raw_allranks model
raw_allranks_fit = fit_dataset_model('raw_allranks')
pd.DataFrame([raw_allranks_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_allranks,3.612215,0.746861


In [18]:
# Fit raw_bronze model
raw_bronze_fit = fit_dataset_model('raw_bronze')
pd.DataFrame([raw_bronze_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_bronze,3.909717,0.792483


In [19]:
# Fit raw_gold model
raw_gold_fit = fit_dataset_model('raw_gold')
pd.DataFrame([raw_gold_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_gold,3.530396,0.742193


In [20]:
# Fit raw_diamond model
raw_diamond_fit = fit_dataset_model('raw_diamond')
pd.DataFrame([raw_diamond_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_diamond,3.659033,0.72531


In [21]:
# Fit raw_gc model
raw_gc_fit = fit_dataset_model('raw_gc')
pd.DataFrame([raw_gc_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_gc,4.052708,0.713095


In [22]:
# Fit diff_allranks model
diff_allranks_fit = fit_dataset_model('diff_allranks')
pd.DataFrame([diff_allranks_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_allranks,2.296026,0.839064


In [23]:
# Fit diff_bronze model
diff_bronze_fit = fit_dataset_model('diff_bronze')
pd.DataFrame([diff_bronze_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_bronze,2.993299,0.840938


In [24]:
# Fit diff_gold model
diff_gold_fit = fit_dataset_model('diff_gold')
pd.DataFrame([diff_gold_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_gold,2.434943,0.822189


In [25]:
# Fit diff_diamond model
diff_diamond_fit = fit_dataset_model('diff_diamond')
pd.DataFrame([diff_diamond_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_diamond,2.366678,0.82233


In [26]:
# Fit diff_gc model
diff_gc_fit = fit_dataset_model('diff_gc')
pd.DataFrame([diff_gc_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_gc,2.613332,0.814993


In [57]:
import json
from pathlib import Path

def save_results(permutation_result, output_dir="../outputs/rfpermute_saved_results/per_dataset", drop_from_memory=False):

    dataset_name = permutation_result["dataset"]
    results_dir = Path(output_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    importance_df = permutation_result["importance_df"]
    null_importances = permutation_result["null_importances"]

    importance_path = results_dir / f"{dataset_name}_importance.csv"
    null_path = results_dir / f"{dataset_name}_null_importances.npy"
    meta_path = results_dir / f"{dataset_name}_meta.json"

    importance_df.to_csv(importance_path, index=False)
    np.save(null_path, null_importances)

    meta = {
        "dataset": dataset_name,
        "n_features": int(importance_df.shape[0]),
        "n_significant_p05": int(importance_df["Significant"].sum()),
        "mean_pct_inc_mse": float(importance_df["PctIncMSE"].mean()),
    }
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    if drop_from_memory:
        permutation_result["null_importances"] = None
        permutation_result["importance_df"] = None

    return {
        "dataset": dataset_name,
        "importance_csv": str(importance_path),
        "null_npy": str(null_path),
        "meta_json": str(meta_path),
    }

In [ ]:
# Optimized alternative implementation (keeps original code unchanged) - copilot was used here to optimize run time. 
# It implemented the _build_oob_index_cache_opt function into the original code
def _build_oob_index_cache_opt(rf, n_samples):
    all_idx = np.arange(n_samples)
    oob_cache = []
    for train_indices in rf.estimators_samples_:
        in_bag = np.zeros(n_samples, dtype=bool)
        in_bag[train_indices] = True
        oob_cache.append(all_idx[~in_bag])
    return oob_cache

def calculate_pct_inc_mse_opt(rf, X, y, n_repeats=2, random_state=123):
    rng = np.random.default_rng(random_state)

    X_np = X.to_numpy()
    y_np = y.to_numpy()
    n_samples, n_features = X_np.shape

    baseline_oob_pred = rf.oob_prediction_
    valid_baseline = ~np.isnan(baseline_oob_pred)

    baseline_mse = mean_squared_error(
        y_np[valid_baseline],
        baseline_oob_pred[valid_baseline]
    )

    oob_cache = _build_oob_index_cache_opt(rf, n_samples)
    all_importances = np.zeros((n_repeats, n_features))

    for repeat in range(n_repeats):
        for j in range(n_features):
            oob_pred_sum = np.zeros(n_samples)
            oob_pred_count = np.zeros(n_samples)

            for tree, oob_indices in zip(rf.estimators_, oob_cache):
                if oob_indices.size == 0:
                    continue

                X_oob = X_np[oob_indices].copy()
                rng.shuffle(X_oob[:, j])

                preds = tree.predict(X_oob)
                oob_pred_sum[oob_indices] += preds
                oob_pred_count[oob_indices] += 1

            valid = oob_pred_count > 0
            permuted_oob_pred = np.zeros(n_samples)
            permuted_oob_pred[valid] = oob_pred_sum[valid] / oob_pred_count[valid]

            permuted_mse = mean_squared_error(
                y_np[valid],
                permuted_oob_pred[valid]
            )

            pct_inc_mse = ((permuted_mse - baseline_mse) / baseline_mse) * 100
            all_importances[repeat, j] = pct_inc_mse
            print(f"Repeat {repeat+1}/{n_repeats}, Feature {j+1}/{n_features} completed")

    mean_importance = all_importances.mean(axis=0)
    sd_importance = all_importances.std(axis=0)

    return mean_importance, sd_importance

def calculate_p_values_opt(X, y, ntree, observed_importance, n_perm=n_permutations,
                           n_repeats_null=n_repeats_NULL, random_state=123):
    rng = np.random.default_rng(random_state)
    null_importances = np.zeros((n_perm, X.shape[1]))

    for i in range(n_perm):
        y_perm = pd.Series(rng.permutation(y.to_numpy()), index=y.index, name=y.name)

        rf_null = RandomForestRegressor(
            n_estimators=ntree,
            max_features=max(1, X.shape[1] // 3),
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=int(rng.integers(0, 123456789)),
        )
        rf_null.fit(X, y_perm)

        null_scores, _ = calculate_pct_inc_mse_opt(
            rf_null,
            X,
            y_perm,
            n_repeats=n_repeats_null,
            random_state=int(rng.integers(0, 123456789)),
        )
        null_importances[i] = null_scores
        print(f"Permutation {i+1}/{n_perm} completed")

    p_values = ((null_importances >= observed_importance).sum(axis=0) + 1) / (n_perm + 1)
    return p_values, null_importances

def run_dataset_permutation_opt(fit_result, n_permutations=n_permutations, n_repeats=n_repeats,
                                n_repeats_null=n_repeats_NULL, random_state=123):
    X = fit_result['X']
    y = fit_result['y']
    rf = fit_result['rf']

    importance_mean, importance_sd = calculate_pct_inc_mse_opt(
        rf,
        X,
        y,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    p_values, null_importances = calculate_p_values_opt(
        X,
        y,
        fit_result['ntree'],
        observed_importance=importance_mean,
        n_perm=n_permutations,
        n_repeats_null=n_repeats_null,
        random_state=random_state
    )

    importance_df = pd.DataFrame({
        'Dataset': fit_result['dataset'],
        'Feature': X.columns,
        'PctIncMSE': importance_mean,
        'PctIncMSE_SD': importance_sd,
        'p_value': p_values,
        'Significant': p_values < 0.05,
    }).sort_values('PctIncMSE', ascending=False).reset_index(drop=True)

    return {
        'dataset': fit_result['dataset'],
        'importance_df': importance_df,
        'null_importances': null_importances,
    }


In [55]:
# Permutation importance for raw_allranks
raw_allranks_perm = run_dataset_permutation_opt(raw_allranks_fit, n_repeats=2, n_repeats_null=1)

Repeat 1/2, Feature 1/28 completed
Repeat 1/2, Feature 2/28 completed
Repeat 1/2, Feature 3/28 completed
Repeat 1/2, Feature 4/28 completed
Repeat 1/2, Feature 5/28 completed
Repeat 1/2, Feature 6/28 completed
Repeat 1/2, Feature 7/28 completed
Repeat 1/2, Feature 8/28 completed
Repeat 1/2, Feature 9/28 completed
Repeat 1/2, Feature 10/28 completed
Repeat 1/2, Feature 11/28 completed
Repeat 1/2, Feature 12/28 completed
Repeat 1/2, Feature 13/28 completed
Repeat 1/2, Feature 14/28 completed
Repeat 1/2, Feature 15/28 completed
Repeat 1/2, Feature 16/28 completed
Repeat 1/2, Feature 17/28 completed
Repeat 1/2, Feature 18/28 completed
Repeat 1/2, Feature 19/28 completed
Repeat 1/2, Feature 20/28 completed
Repeat 1/2, Feature 21/28 completed
Repeat 1/2, Feature 22/28 completed
Repeat 1/2, Feature 23/28 completed
Repeat 1/2, Feature 24/28 completed
Repeat 1/2, Feature 25/28 completed
Repeat 1/2, Feature 26/28 completed
Repeat 1/2, Feature 27/28 completed
Repeat 1/2, Feature 28/28 completed
R

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_allranks,shots.conceded.ag,115.226590,0.019495,0.047619,True
1,raw_allranks,percentage.behind.ball,53.539950,0.043094,0.047619,True
2,raw_allranks,shots.ag,42.904714,0.009772,0.047619,True
3,raw_allranks,saves.ag,18.955434,0.046307,0.047619,True
4,raw_allranks,percentage.defensive.third,1.925364,0.026281,0.047619,True
5,raw_allranks,percentage.offensive.third,1.071704,0.008274,0.047619,True
6,raw_allranks,demos.inflicted.ag,0.671358,0.006302,0.047619,True
7,raw_allranks,count.stolen.small.pads.ag,0.520989,0.012964,0.047619,True
8,raw_allranks,avg.boost.amount,0.454756,0.011876,0.047619,True
9,raw_allranks,percentage.supersonic.speed,0.428648,0.008135,0.047619,True


In [ ]:
raw_allranks_perm['importance_df'].head(10)

{'dataset': 'raw_allranks',
 'importance_csv': '..\\outputs\\rfpermute_saved_results\\per_dataset\\raw_allranks_importance.csv',
 'null_npy': '..\\outputs\\rfpermute_saved_results\\per_dataset\\raw_allranks_null_importances.npy',
 'meta_json': '..\\outputs\\rfpermute_saved_results\\per_dataset\\raw_allranks_meta.json'}

In [ ]:
save_results(raw_allranks_perm)
del raw_allranks_perm

In [60]:
# Permutation importance for raw_bronze
raw_bronze_perm = run_dataset_permutation_opt(raw_bronze_fit)

Repeat 1/2, Feature 1/28 completed
Repeat 1/2, Feature 2/28 completed
Repeat 1/2, Feature 3/28 completed
Repeat 1/2, Feature 4/28 completed
Repeat 1/2, Feature 5/28 completed
Repeat 1/2, Feature 6/28 completed
Repeat 1/2, Feature 7/28 completed
Repeat 1/2, Feature 8/28 completed
Repeat 1/2, Feature 9/28 completed
Repeat 1/2, Feature 10/28 completed
Repeat 1/2, Feature 11/28 completed
Repeat 1/2, Feature 12/28 completed
Repeat 1/2, Feature 13/28 completed
Repeat 1/2, Feature 14/28 completed
Repeat 1/2, Feature 15/28 completed
Repeat 1/2, Feature 16/28 completed
Repeat 1/2, Feature 17/28 completed
Repeat 1/2, Feature 18/28 completed
Repeat 1/2, Feature 19/28 completed
Repeat 1/2, Feature 20/28 completed
Repeat 1/2, Feature 21/28 completed
Repeat 1/2, Feature 22/28 completed
Repeat 1/2, Feature 23/28 completed
Repeat 1/2, Feature 24/28 completed
Repeat 1/2, Feature 25/28 completed
Repeat 1/2, Feature 26/28 completed
Repeat 1/2, Feature 27/28 completed
Repeat 1/2, Feature 28/28 completed
R

In [ ]:
raw_bronze_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_bronze,shots.conceded.ag,94.954890,0.393052,0.047619,True
1,raw_bronze,shots.ag,71.158366,0.218503,0.047619,True
2,raw_bronze,percentage.behind.ball,62.048959,0.076564,0.047619,True
3,raw_bronze,saves.ag,10.180133,0.132544,0.047619,True
4,raw_bronze,percentage.offensive.third,3.210444,0.390876,0.047619,True
5,raw_bronze,percentage.defensive.third,1.833549,0.092545,0.047619,True
6,raw_bronze,100.boost.time.ag,1.158725,0.017504,0.047619,True
7,raw_bronze,percentage.high.in.air,1.022066,0.079980,0.047619,True
8,raw_bronze,true.wastage,0.620564,0.042481,0.047619,True
9,raw_bronze,real.speed.ag,0.590549,0.062039,0.047619,True


In [63]:
save_results(raw_bronze_perm)
del raw_bronze_perm

In [64]:
# Permutation importance for raw_gold
raw_gold_perm = run_dataset_permutation_opt(raw_gold_fit)

Repeat 1/2, Feature 1/28 completed
Repeat 1/2, Feature 2/28 completed
Repeat 1/2, Feature 3/28 completed
Repeat 1/2, Feature 4/28 completed
Repeat 1/2, Feature 5/28 completed
Repeat 1/2, Feature 6/28 completed
Repeat 1/2, Feature 7/28 completed
Repeat 1/2, Feature 8/28 completed
Repeat 1/2, Feature 9/28 completed
Repeat 1/2, Feature 10/28 completed
Repeat 1/2, Feature 11/28 completed
Repeat 1/2, Feature 12/28 completed
Repeat 1/2, Feature 13/28 completed
Repeat 1/2, Feature 14/28 completed
Repeat 1/2, Feature 15/28 completed
Repeat 1/2, Feature 16/28 completed
Repeat 1/2, Feature 17/28 completed
Repeat 1/2, Feature 18/28 completed
Repeat 1/2, Feature 19/28 completed
Repeat 1/2, Feature 20/28 completed
Repeat 1/2, Feature 21/28 completed
Repeat 1/2, Feature 22/28 completed
Repeat 1/2, Feature 23/28 completed
Repeat 1/2, Feature 24/28 completed
Repeat 1/2, Feature 25/28 completed
Repeat 1/2, Feature 26/28 completed
Repeat 1/2, Feature 27/28 completed
Repeat 1/2, Feature 28/28 completed
R

In [65]:
raw_gold_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_gold,shots.conceded.ag,92.538715,0.309705,0.047619,True
1,raw_gold,shots.ag,47.634140,0.111236,0.047619,True
2,raw_gold,percentage.behind.ball,42.552901,0.044463,0.047619,True
3,raw_gold,saves.ag,16.822619,0.168059,0.047619,True
4,raw_gold,percentage.defensive.third,1.003097,0.076801,0.047619,True
5,raw_gold,percentage.offensive.third,0.851362,0.142022,0.047619,True
6,raw_gold,avg.boost.amount,0.419519,0.080932,0.047619,True
7,raw_gold,percentage.high.in.air,0.392029,0.043651,0.047619,True
8,raw_gold,100.boost.time.ag,0.262282,0.004328,0.047619,True
9,raw_gold,true.wastage,0.208204,0.034885,0.047619,True


In [66]:
save_results(raw_gold_perm)
del raw_gold_perm

In [67]:
# Permutation importance for raw_diamond
raw_diamond_perm = run_dataset_permutation_opt(raw_diamond_fit)

Repeat 1/2, Feature 1/28 completed
Repeat 1/2, Feature 2/28 completed
Repeat 1/2, Feature 3/28 completed
Repeat 1/2, Feature 4/28 completed
Repeat 1/2, Feature 5/28 completed
Repeat 1/2, Feature 6/28 completed
Repeat 1/2, Feature 7/28 completed
Repeat 1/2, Feature 8/28 completed
Repeat 1/2, Feature 9/28 completed
Repeat 1/2, Feature 10/28 completed
Repeat 1/2, Feature 11/28 completed
Repeat 1/2, Feature 12/28 completed
Repeat 1/2, Feature 13/28 completed
Repeat 1/2, Feature 14/28 completed
Repeat 1/2, Feature 15/28 completed
Repeat 1/2, Feature 16/28 completed
Repeat 1/2, Feature 17/28 completed
Repeat 1/2, Feature 18/28 completed
Repeat 1/2, Feature 19/28 completed
Repeat 1/2, Feature 20/28 completed
Repeat 1/2, Feature 21/28 completed
Repeat 1/2, Feature 22/28 completed
Repeat 1/2, Feature 23/28 completed
Repeat 1/2, Feature 24/28 completed
Repeat 1/2, Feature 25/28 completed
Repeat 1/2, Feature 26/28 completed
Repeat 1/2, Feature 27/28 completed
Repeat 1/2, Feature 28/28 completed
R

In [68]:
raw_diamond_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_diamond,shots.conceded.ag,66.281061,0.198261,0.047619,True
1,raw_diamond,percentage.behind.ball,60.783436,0.013545,0.047619,True
2,raw_diamond,shots.ag,43.333653,0.156806,0.047619,True
3,raw_diamond,saves.ag,14.731173,0.003540,0.047619,True
4,raw_diamond,percentage.defensive.third,1.593452,0.080896,0.047619,True
5,raw_diamond,demos.inflicted.ag,0.987174,0.055803,0.047619,True
6,raw_diamond,percentage.high.in.air,0.624119,0.011673,0.047619,True
7,raw_diamond,percentage.offensive.third,0.563054,0.042465,0.047619,True
8,raw_diamond,percentage.on.ground,0.394003,0.038889,0.047619,True
9,raw_diamond,avg.boost.amount,0.379493,0.003345,0.047619,True


In [69]:
save_results(raw_diamond_perm)
del raw_diamond_perm

In [70]:
# Permutation importance for raw_gc
raw_gc_perm = run_dataset_permutation_opt(raw_gc_fit)

Repeat 1/2, Feature 1/27 completed
Repeat 1/2, Feature 2/27 completed
Repeat 1/2, Feature 3/27 completed
Repeat 1/2, Feature 4/27 completed
Repeat 1/2, Feature 5/27 completed
Repeat 1/2, Feature 6/27 completed
Repeat 1/2, Feature 7/27 completed
Repeat 1/2, Feature 8/27 completed
Repeat 1/2, Feature 9/27 completed
Repeat 1/2, Feature 10/27 completed
Repeat 1/2, Feature 11/27 completed
Repeat 1/2, Feature 12/27 completed
Repeat 1/2, Feature 13/27 completed
Repeat 1/2, Feature 14/27 completed
Repeat 1/2, Feature 15/27 completed
Repeat 1/2, Feature 16/27 completed
Repeat 1/2, Feature 17/27 completed
Repeat 1/2, Feature 18/27 completed
Repeat 1/2, Feature 19/27 completed
Repeat 1/2, Feature 20/27 completed
Repeat 1/2, Feature 21/27 completed
Repeat 1/2, Feature 22/27 completed
Repeat 1/2, Feature 23/27 completed
Repeat 1/2, Feature 24/27 completed
Repeat 1/2, Feature 25/27 completed
Repeat 1/2, Feature 26/27 completed
Repeat 1/2, Feature 27/27 completed
Repeat 2/2, Feature 1/27 completed
Re

In [71]:
raw_gc_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_gc,percentage.behind.ball,74.756793,0.406356,0.047619,True
1,raw_gc,shots.conceded.ag,57.553189,0.114803,0.047619,True
2,raw_gc,shots.ag,33.586687,0.146254,0.047619,True
3,raw_gc,saves.ag,11.562005,0.008483,0.047619,True
4,raw_gc,demos.inflicted.ag,1.345313,0.029993,0.047619,True
5,raw_gc,percentage.defensive.third,1.320407,0.037088,0.047619,True
6,raw_gc,percentage.on.ground,0.819846,0.053162,0.047619,True
7,raw_gc,percentage.offensive.third,0.677165,0.066742,0.047619,True
8,raw_gc,avg.boost.amount,0.486456,0.004694,0.047619,True
9,raw_gc,count.stolen.small.pads.ag,0.380627,0.025366,0.047619,True


In [72]:
save_results(raw_gc_perm)
del raw_gc_perm

In [73]:
# Permutation importance for diff_allranks
diff_allranks_perm = run_dataset_permutation_opt(diff_allranks_fit)

Repeat 1/2, Feature 1/26 completed
Repeat 1/2, Feature 2/26 completed
Repeat 1/2, Feature 3/26 completed
Repeat 1/2, Feature 4/26 completed
Repeat 1/2, Feature 5/26 completed
Repeat 1/2, Feature 6/26 completed
Repeat 1/2, Feature 7/26 completed
Repeat 1/2, Feature 8/26 completed
Repeat 1/2, Feature 9/26 completed
Repeat 1/2, Feature 10/26 completed
Repeat 1/2, Feature 11/26 completed
Repeat 1/2, Feature 12/26 completed
Repeat 1/2, Feature 13/26 completed
Repeat 1/2, Feature 14/26 completed
Repeat 1/2, Feature 15/26 completed
Repeat 1/2, Feature 16/26 completed
Repeat 1/2, Feature 17/26 completed
Repeat 1/2, Feature 18/26 completed
Repeat 1/2, Feature 19/26 completed
Repeat 1/2, Feature 20/26 completed
Repeat 1/2, Feature 21/26 completed
Repeat 1/2, Feature 22/26 completed
Repeat 1/2, Feature 23/26 completed
Repeat 1/2, Feature 24/26 completed
Repeat 1/2, Feature 25/26 completed
Repeat 1/2, Feature 26/26 completed
Repeat 2/2, Feature 1/26 completed
Repeat 2/2, Feature 2/26 completed
Rep

In [74]:
diff_allranks_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_allranks,diff.shots.ag,278.089835,0.203559,0.047619,True
1,diff_allranks,diff.percentage.behind.ball.ag,112.297884,0.092606,0.047619,True
2,diff_allranks,diff.saves.ag,55.326043,0.065539,0.047619,True
3,diff_allranks,diff.percentage.defensive.third.ag,2.393120,0.048858,0.047619,True
4,diff_allranks,diff.percentage.offensive.third.ag,1.654305,0.007177,0.047619,True
5,diff_allranks,diff.percentage.high.in.air.ag,0.936369,0.071427,0.047619,True
6,diff_allranks,diff.demos.inflicted.ag,0.913169,0.001204,0.047619,True
7,diff_allranks,diff.amount.stolen.ag,0.663848,0.004970,0.047619,True
8,diff_allranks,diff.count.stolen.small.pads.ag,0.613274,0.015623,0.047619,True
9,diff_allranks,diff.real.speed.ag,0.497254,0.008689,0.047619,True


In [75]:
save_results(diff_allranks_perm)
del diff_allranks_perm

In [76]:
# Permutation importance for diff_bronze
diff_bronze_perm = run_dataset_permutation_opt(diff_bronze_fit)

Repeat 1/2, Feature 1/26 completed
Repeat 1/2, Feature 2/26 completed
Repeat 1/2, Feature 3/26 completed
Repeat 1/2, Feature 4/26 completed
Repeat 1/2, Feature 5/26 completed
Repeat 1/2, Feature 6/26 completed
Repeat 1/2, Feature 7/26 completed
Repeat 1/2, Feature 8/26 completed
Repeat 1/2, Feature 9/26 completed
Repeat 1/2, Feature 10/26 completed
Repeat 1/2, Feature 11/26 completed
Repeat 1/2, Feature 12/26 completed
Repeat 1/2, Feature 13/26 completed
Repeat 1/2, Feature 14/26 completed
Repeat 1/2, Feature 15/26 completed
Repeat 1/2, Feature 16/26 completed
Repeat 1/2, Feature 17/26 completed
Repeat 1/2, Feature 18/26 completed
Repeat 1/2, Feature 19/26 completed
Repeat 1/2, Feature 20/26 completed
Repeat 1/2, Feature 21/26 completed
Repeat 1/2, Feature 22/26 completed
Repeat 1/2, Feature 23/26 completed
Repeat 1/2, Feature 24/26 completed
Repeat 1/2, Feature 25/26 completed
Repeat 1/2, Feature 26/26 completed
Repeat 2/2, Feature 1/26 completed
Repeat 2/2, Feature 2/26 completed
Rep

In [77]:
diff_bronze_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_bronze,diff.shots.ag,235.999129,0.289890,0.047619,True
1,diff_bronze,diff.percentage.behind.ball.ag,109.875988,0.057687,0.047619,True
2,diff_bronze,diff.saves.ag,24.066700,0.103499,0.047619,True
3,diff_bronze,diff.percentage.offensive.third.ag,3.380427,0.057649,0.047619,True
4,diff_bronze,diff.percentage.defensive.third.ag,2.031156,0.029793,0.047619,True
5,diff_bronze,diff.percentage.high.in.air.ag,1.594342,0.004475,0.047619,True
6,diff_bronze,diff.amount.stolen.ag,0.933654,0.047686,0.047619,True
7,diff_bronze,diff.count.powerslide.ag,0.714457,0.005348,0.047619,True
8,diff_bronze,diff.count.stolen.small.pads.ag,0.623851,0.051412,0.047619,True
9,diff_bronze,diff.percentage.supersonic.speed.ag,0.397655,0.011411,0.047619,True


In [78]:
save_results(diff_bronze_perm)
del diff_bronze_perm

In [79]:
# Permutation importance for diff_gold
diff_gold_perm = run_dataset_permutation_opt(diff_gold_fit)

Repeat 1/2, Feature 1/26 completed
Repeat 1/2, Feature 2/26 completed
Repeat 1/2, Feature 3/26 completed
Repeat 1/2, Feature 4/26 completed
Repeat 1/2, Feature 5/26 completed
Repeat 1/2, Feature 6/26 completed
Repeat 1/2, Feature 7/26 completed
Repeat 1/2, Feature 8/26 completed
Repeat 1/2, Feature 9/26 completed
Repeat 1/2, Feature 10/26 completed
Repeat 1/2, Feature 11/26 completed
Repeat 1/2, Feature 12/26 completed
Repeat 1/2, Feature 13/26 completed
Repeat 1/2, Feature 14/26 completed
Repeat 1/2, Feature 15/26 completed
Repeat 1/2, Feature 16/26 completed
Repeat 1/2, Feature 17/26 completed
Repeat 1/2, Feature 18/26 completed
Repeat 1/2, Feature 19/26 completed
Repeat 1/2, Feature 20/26 completed
Repeat 1/2, Feature 21/26 completed
Repeat 1/2, Feature 22/26 completed
Repeat 1/2, Feature 23/26 completed
Repeat 1/2, Feature 24/26 completed
Repeat 1/2, Feature 25/26 completed
Repeat 1/2, Feature 26/26 completed
Repeat 2/2, Feature 1/26 completed
Repeat 2/2, Feature 2/26 completed
Rep

In [80]:
diff_gold_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_gold,diff.shots.ag,264.062592,0.204000,0.047619,True
1,diff_gold,diff.percentage.behind.ball.ag,84.739153,0.155185,0.047619,True
2,diff_gold,diff.saves.ag,42.676216,0.117698,0.047619,True
3,diff_gold,diff.percentage.defensive.third.ag,1.759558,0.022124,0.047619,True
4,diff_gold,diff.percentage.offensive.third.ag,1.549799,0.098514,0.047619,True
5,diff_gold,diff.percentage.high.in.air.ag,1.188292,0.029963,0.047619,True
6,diff_gold,diff.amount.stolen.ag,0.807082,0.018015,0.047619,True
7,diff_gold,diff.demos.inflicted.ag,0.453477,0.031985,0.047619,True
8,diff_gold,diff.count.stolen.small.pads.ag,0.451906,0.031835,0.047619,True
9,diff_gold,diff.count.stolen.big.pads.ag,0.372909,0.009862,0.047619,True


In [81]:
save_results(diff_gold_perm)
del diff_gold_perm

In [82]:
# Permutation importance for diff_diamond
diff_diamond_perm = run_dataset_permutation_opt(diff_diamond_fit)

Repeat 1/2, Feature 1/26 completed
Repeat 1/2, Feature 2/26 completed
Repeat 1/2, Feature 3/26 completed
Repeat 1/2, Feature 4/26 completed
Repeat 1/2, Feature 5/26 completed
Repeat 1/2, Feature 6/26 completed
Repeat 1/2, Feature 7/26 completed
Repeat 1/2, Feature 8/26 completed
Repeat 1/2, Feature 9/26 completed
Repeat 1/2, Feature 10/26 completed
Repeat 1/2, Feature 11/26 completed
Repeat 1/2, Feature 12/26 completed
Repeat 1/2, Feature 13/26 completed
Repeat 1/2, Feature 14/26 completed
Repeat 1/2, Feature 15/26 completed
Repeat 1/2, Feature 16/26 completed
Repeat 1/2, Feature 17/26 completed
Repeat 1/2, Feature 18/26 completed
Repeat 1/2, Feature 19/26 completed
Repeat 1/2, Feature 20/26 completed
Repeat 1/2, Feature 21/26 completed
Repeat 1/2, Feature 22/26 completed
Repeat 1/2, Feature 23/26 completed
Repeat 1/2, Feature 24/26 completed
Repeat 1/2, Feature 25/26 completed
Repeat 1/2, Feature 26/26 completed
Repeat 2/2, Feature 1/26 completed
Repeat 2/2, Feature 2/26 completed
Rep

In [83]:
diff_diamond_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_diamond,diff.shots.ag,218.378016,0.475251,0.047619,True
1,diff_diamond,diff.percentage.behind.ball.ag,124.908040,0.051170,0.047619,True
2,diff_diamond,diff.saves.ag,44.860182,0.061885,0.047619,True
3,diff_diamond,diff.percentage.defensive.third.ag,2.278121,0.009033,0.047619,True
4,diff_diamond,diff.percentage.offensive.third.ag,1.788719,0.032000,0.047619,True
5,diff_diamond,diff.demos.inflicted.ag,0.948690,0.109201,0.047619,True
6,diff_diamond,diff.amount.stolen.ag,0.711726,0.047684,0.047619,True
7,diff_diamond,diff.percentage.high.in.air.ag,0.633403,0.015427,0.047619,True
8,diff_diamond,diff.count.stolen.small.pads.ag,0.566906,0.031757,0.047619,True
9,diff_diamond,diff.real.speed.ag,0.169049,0.034410,0.047619,True


In [84]:
save_results(diff_diamond_perm)
del diff_diamond_perm

In [85]:
# Permutation importance for diff_gc
diff_gc_perm = run_dataset_permutation_opt(diff_gc_fit)

Repeat 1/2, Feature 1/26 completed
Repeat 1/2, Feature 2/26 completed
Repeat 1/2, Feature 3/26 completed
Repeat 1/2, Feature 4/26 completed
Repeat 1/2, Feature 5/26 completed
Repeat 1/2, Feature 6/26 completed
Repeat 1/2, Feature 7/26 completed
Repeat 1/2, Feature 8/26 completed
Repeat 1/2, Feature 9/26 completed
Repeat 1/2, Feature 10/26 completed
Repeat 1/2, Feature 11/26 completed
Repeat 1/2, Feature 12/26 completed
Repeat 1/2, Feature 13/26 completed
Repeat 1/2, Feature 14/26 completed
Repeat 1/2, Feature 15/26 completed
Repeat 1/2, Feature 16/26 completed
Repeat 1/2, Feature 17/26 completed
Repeat 1/2, Feature 18/26 completed
Repeat 1/2, Feature 19/26 completed
Repeat 1/2, Feature 20/26 completed
Repeat 1/2, Feature 21/26 completed
Repeat 1/2, Feature 22/26 completed
Repeat 1/2, Feature 23/26 completed
Repeat 1/2, Feature 24/26 completed
Repeat 1/2, Feature 25/26 completed
Repeat 1/2, Feature 26/26 completed
Repeat 2/2, Feature 1/26 completed
Repeat 2/2, Feature 2/26 completed
Rep

In [86]:
diff_gc_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_gc,diff.shots.ag,177.831700,0.234321,0.047619,True
1,diff_gc,diff.percentage.behind.ball.ag,149.354590,0.383516,0.047619,True
2,diff_gc,diff.saves.ag,35.531561,0.128938,0.047619,True
3,diff_gc,diff.percentage.offensive.third.ag,2.166800,0.000034,0.047619,True
4,diff_gc,diff.percentage.defensive.third.ag,1.675983,0.060102,0.047619,True
5,diff_gc,diff.demos.inflicted.ag,1.395569,0.029803,0.047619,True
6,diff_gc,diff.count.stolen.small.pads.ag,0.600733,0.013159,0.047619,True
7,diff_gc,diff.amount.stolen.ag,0.584673,0.024321,0.047619,True
8,diff_gc,diff.real.speed.ag,0.362455,0.031940,0.047619,True
9,diff_gc,diff.percentage.on.ground.ag,0.283625,0.047409,0.047619,True


In [87]:
save_results(diff_gc_perm)
del diff_gc_perm

In [88]:
results_dir = Path("../outputs/rfpermute_saved_results")
per_dataset_dir = results_dir / "per_dataset"
results_dir.mkdir(parents=True, exist_ok=True)
per_dataset_dir.mkdir(parents=True, exist_ok=True)

fit_results = [
    raw_allranks_fit,
    raw_bronze_fit,
    raw_gold_fit,
    raw_diamond_fit,
    raw_gc_fit,
    diff_allranks_fit,
    diff_bronze_fit,
    diff_gold_fit,
    diff_diamond_fit,
    diff_gc_fit,
]

model_summary_df = pd.DataFrame([result["model_summary"] for result in fit_results])

importance_frames = []
for fit in fit_results:
    dataset = fit["dataset"]
    importance_file = per_dataset_dir / f"{dataset}_importance.csv"
    if not importance_file.exists():
        raise FileNotFoundError(f"Missing saved importance file: {importance_file}")
    importance_frames.append(pd.read_csv(importance_file))

all_importance_df = pd.concat(importance_frames, ignore_index=True)
significance_summary = all_importance_df.groupby("Dataset")["Significant"].sum().reset_index(name="N_Significant_p05")
model_summary_df = model_summary_df.merge(significance_summary, on="Dataset", how="left")

model_summary_path = results_dir / "rfpermute_model_summary.csv"
importance_path = results_dir / "rfpermute_importance_results.csv"
json_path = results_dir / "rfpermute_results.json"

model_summary_df.to_csv(model_summary_path, index=False)
all_importance_df.to_csv(importance_path, index=False)

results_bundle = {
    "model_summary": model_summary_df.to_dict(orient="records"),
    "feature_importance": all_importance_df.to_dict(orient="records"),
    "datasets": {},
}

with json_path.open("w", encoding="utf-8") as file_handle:
    json.dump(results_bundle, file_handle, indent=2)

print(f"Saved model summary: {model_summary_path}")
print(f"Saved feature importance: {importance_path}")
print(f"Saved json bundle: {json_path}")

model_summary_df

Saved model summary: ..\outputs\rfpermute_saved_results\rfpermute_model_summary.csv
Saved feature importance: ..\outputs\rfpermute_saved_results\rfpermute_importance_results.csv
Saved json bundle: ..\outputs\rfpermute_saved_results\rfpermute_results.json


,Dataset,OOB_MSE,OOB_R2,N_Significant_p05
0,raw_allranks,3.612215,0.746861,23
1,raw_bronze,3.909717,0.792483,20
2,raw_gold,3.530396,0.742193,14
3,raw_diamond,3.659033,0.725310,19
4,raw_gc,4.052708,0.713095,14
5,diff_allranks,2.296026,0.839064,15
6,diff_bronze,2.993299,0.840938,14
7,diff_gold,2.434943,0.822189,16
8,diff_diamond,2.366678,0.822330,11
9,diff_gc,2.613332,0.814993,14
